#**Two-Tower Neural Network for Candidate Generation**

In [ ]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras.layers import Input, Dense, Dot, Concatenate
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

In [ ]:
customers = pd.read_csv('customers.csv')
policies = pd.read_csv('policies.csv')
interactions = pd.read_csv('interactions.csv')

In [ ]:
# Merge customer and policy data into the interactions dataframe.
data = interactions.merge(customers, on='customer_id', how='left') \
                   .merge(policies, on='policy_id', how='left')

In [ ]:
data['interaction_score'] = (
    (data['purchased'] * 5) +  # Strongest signal
    (data['abandoned_cart'] * 3) +  # Indicates strong intent
    (data['clicked'] * 2) +  # Clicks indicate interest
    (data['viewed_duration'] / 30) +  # Normalize by 30 seconds
    (data['comparison_count'] * 1)  # Weaker signal
)

In [ ]:
data['label'] = data['interaction_score'] / data['interaction_score'].max()

In [ ]:
customer_numeric_cols = ['age', 'policy_ownership_count', 'credit_score']
customer_categorical_cols = ['gender', 'income_bracket', 'employment_status', 'marital_status', 'location_city', 'preferred_policy_type']


In [ ]:
# For the policy tower, we pick key numeric and categorical policy features.
policy_numeric_cols = ['sum_assured (INR)', 'premium_amount (INR)', 'policy_duration_years']
policy_categorical_cols = ['policy_type', 'risk_category', 'customer_target_group']

In [ ]:
# Interaction features
interaction_numeric_cols = ['clicked', 'viewed_duration', 'comparison_count', 'abandoned_cart']

In [ ]:
# Extract feature subsets
customer_features_df = data[customer_numeric_cols + customer_categorical_cols]
policy_features_df = data[policy_numeric_cols + policy_categorical_cols]
interaction_features_df = data[interaction_numeric_cols]

labels = data['label'].values

In [ ]:
for column in ['sum_assured (INR)', 'premium_amount (INR)']:
    policy_features_df[column] = policy_features_df[column].str.replace(',', '').astype(float)

<ipython-input-12-299fafa69a5c>:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  policy_features_df[column] = policy_features_df[column].str.replace(',', '').astype(float)
<ipython-input-12-299fafa69a5c>:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  policy_features_df[column] = policy_features_df[column].str.replace(',', '').astype(float)


In [ ]:
customer_preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), customer_numeric_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore'), customer_categorical_cols)
    ]
)

In [ ]:
policy_preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), policy_numeric_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore'), policy_categorical_cols)
    ]
)

In [ ]:
interaction_preprocessor = StandardScaler()

customer_features = customer_preprocessor.fit_transform(customer_features_df)
policy_features = policy_preprocessor.fit_transform(policy_features_df)
interaction_features = interaction_preprocessor.fit_transform(interaction_features_df)

In [ ]:
X_cust_train, X_cust_test, X_policy_train, X_policy_test, X_interact_train, X_interact_test, y_train, y_test = train_test_split(
    customer_features, policy_features, interaction_features, labels, test_size=0.2, random_state=42
)

In [ ]:
customer_input = Input(shape=(X_cust_train.shape[1],), name='customer_input')
policy_input = Input(shape=(X_policy_train.shape[1],), name='policy_input')
interaction_input = Input(shape=(X_interact_train.shape[1],), name='interaction_input')


In [ ]:
# Customer tower
cust_dense = Dense(64, activation='relu')(customer_input)
cust_embed = Dense(32, activation='relu', name='customer_embedding')(cust_dense)

In [ ]:
# Policy tower
policy_dense = Dense(64, activation='relu')(policy_input)
policy_embed = Dense(32, activation='relu', name='policy_embedding')(policy_dense)

In [ ]:
# Interaction layer - This acts as an additional feature to influence the match score
interaction_dense = Dense(16, activation='relu')(interaction_input)

In [ ]:
# Combine customer, policy, and interaction layers
combined = Concatenate()([cust_embed, policy_embed, interaction_dense])

# Final prediction layer
output = Dense(1, activation='sigmoid')(combined)

In [ ]:
# Compile model
model = Model(inputs=[customer_input, policy_input, interaction_input], outputs=output)
model.compile(optimizer=Adam(learning_rate=0.001), loss='binary_crossentropy', metrics=['accuracy'])

model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)              ┃ Output Shape           ┃        Param # ┃ Connected to           ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━┩
│ customer_input            │ (None, 36)             │              0 │ -                      │
│ (InputLayer)              │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ policy_input (InputLayer) │ (None, 44)             │              0 │ -                      │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ dense (Dense)             │ (None, 64)             │          2,368 │ customer_input[0][0]   │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ dense_1 (Dense)           │ (None, 64)             │          2,880 │ policy_input[0][0]     │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ interaction_input         │ (None, 4)              │              0 │ -                      │
│ (InputLayer)              │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ customer_embedding        │ (None, 32)             │          2,080 │ dense[0][0]            │
│ (Dense)                   │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ policy_embedding (Dense)  │ (None, 32)             │          2,080 │ dense_1[0][0]          │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ dense_2 (Dense)           │ (None, 16)             │             80 │ interaction_input[0][… │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ concatenate (Concatenate) │ (None, 80)             │              0 │ customer_embedding[0]… │
│                           │                        │                │ policy_embedding[0][0… │
│                           │                        │                │ dense_2[0][0]          │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ dense_3 (Dense)           │ (None, 1)              │             81 │ concatenate[0][0]      │
└───────────────────────────┴────────────────────────┴────────────────┴────────────────────────┘

 Total params: 9,569 (37.38 KB)

 Trainable params: 9,569 (37.38 KB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
history = model.fit(
    [X_cust_train, X_policy_train, X_interact_train], y_train,
    validation_split=0.1,
    epochs=10,
    batch_size=32
)

Epoch 1/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 3s 306ms/step - accuracy: 0.1389 - loss: 0.6968 - val_accuracy: 0.1250 - val_loss: 0.7884
Epoch 2/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 1s 111ms/step - accuracy: 0.2331 - loss: 0.6655 - val_accuracy: 0.1250 - val_loss: 0.8032
Epoch 3/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 109ms/step - accuracy: 0.2118 - loss: 0.6427 - val_accuracy: 0.1250 - val_loss: 0.8187
Epoch 4/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 112ms/step - accuracy: 0.2196 - loss: 0.6351 - val_accuracy: 0.1250 - val_loss: 0.8312
Epoch 5/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 1s 105ms/step - accuracy: 0.2626 - loss: 0.6065 - val_accuracy: 0.1250 - val_loss: 0.8423
Epoch 6/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 117ms/step - accuracy: 0.2352 - loss: 0.6180 - val_accuracy: 0.1250 - val_loss: 0.8444
Epoch 7/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 111ms/step - accuracy: 0.2196 - loss: 0.6171 - val_accuracy: 0.1250 - val_loss: 0.8449
Epoch 8/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 108ms/step - accuracy: 0.2509 - loss: 0.5889 - val_accuracy: 0.1250 - val_loss:

In [ ]:
eval_results = model.evaluate([X_cust_test, X_policy_test, X_interact_test], y_test)
print(f"Test Loss: {eval_results[0]:.4f} | Test Accuracy: {eval_results[1]:.4f}")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 214ms/step - accuracy: 0.2000 - loss: 0.6414
Test Loss: 0.6414 | Test Accuracy: 0.2000


In [ ]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras.layers import Input, Dense, Dot, Concatenate, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.utils import class_weight

# ------------------------
# 1. Data Loading & Merging
# ------------------------
customers = pd.read_csv('customers.csv')
policies = pd.read_csv('policies.csv')
interactions = pd.read_csv('interactions.csv')

data = interactions.merge(customers, on='customer_id', how='left') \
                   .merge(policies, on='policy_id', how='left')

# ------------------------
# 2. Label Creation & Analysis
# ------------------------
# Check purchase distribution (assumed to be a strong positive signal)
print("Purchase distribution:")
print(data['purchased'].value_counts())

# Use the 'purchased' flag as the label for binary classification.
data['label'] = data['purchased']

# Alternatively, if you prefer a composite score, you might do:
# data['interaction_score'] = (
#     (data['purchased'] * 5) +
#     (data['abandoned_cart'] * 3) +
#     (data['clicked'] * 2) +
#     (data['viewed_duration'] / 30) +
#     (data['comparison_count'] * 1)
# )
# # Use a threshold (for example, median) to binarize.
# threshold = data['interaction_score'].median()
# data['label'] = (data['interaction_score'] > threshold).astype(int)

# ------------------------
# 3. Feature Selection & Preprocessing
# ------------------------
# Define feature columns.
customer_numeric_cols = ['age', 'policy_ownership_count', 'credit_score']
customer_categorical_cols = ['gender', 'income_bracket', 'employment_status',
                             'marital_status', 'location_city', 'preferred_policy_type']

policy_numeric_cols = ['sum_assured (INR)', 'premium_amount (INR)', 'policy_duration_years']
policy_categorical_cols = ['policy_type', 'risk_category', 'customer_target_group']

interaction_numeric_cols = ['clicked', 'viewed_duration', 'comparison_count', 'abandoned_cart']

# Extract dataframes.
customer_features_df = data[customer_numeric_cols + customer_categorical_cols]
policy_features_df = data[policy_numeric_cols + policy_categorical_cols]
interaction_features_df = data[interaction_numeric_cols]

labels = data['label'].values

# Clean numeric columns for policies.
for column in ['sum_assured (INR)', 'premium_amount (INR)']:
    policy_features_df[column] = policy_features_df[column].str.replace(',', '').astype(float)

# Preprocessor for customer features.
customer_preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), customer_numeric_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore'), customer_categorical_cols)
    ]
)

# Preprocessor for policy features.
policy_preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), policy_numeric_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore'), policy_categorical_cols)
    ]
)

# Standardize interaction features.
interaction_preprocessor = StandardScaler()

customer_features = customer_preprocessor.fit_transform(customer_features_df)
policy_features = policy_preprocessor.fit_transform(policy_features_df)
interaction_features = interaction_preprocessor.fit_transform(interaction_features_df)

# ------------------------
# 4. Train-Test Split
# ------------------------
X_cust_train, X_cust_test, X_policy_train, X_policy_test, X_interact_train, X_interact_test, y_train, y_test = train_test_split(
    customer_features, policy_features, interaction_features, labels, test_size=0.2, random_state=42
)

# ------------------------
# 5. Model Definition
# ------------------------
# Define model inputs.
customer_input = Input(shape=(X_cust_train.shape[1],), name='customer_input')
interaction_input = Input(shape=(X_interact_train.shape[1],), name='interaction_input')
policy_input = Input(shape=(X_policy_train.shape[1],), name='policy_input')

# Query Tower: Merge customer profile and interaction signals.
cust_dense = Dense(64, activation='relu')(customer_input)
cust_dense = Dropout(0.2)(cust_dense)
interact_dense = Dense(16, activation='relu')(interaction_input)
interact_dense = Dropout(0.2)(interact_dense)
query_concat = Concatenate()([cust_dense, interact_dense])
query_embed = Dense(32, activation='relu', name='query_embedding')(query_concat)

# Candidate Tower: Policy features.
policy_dense = Dense(64, activation='relu')(policy_input)
policy_dense = Dropout(0.2)(policy_dense)
policy_embed = Dense(32, activation='relu', name='policy_embedding')(policy_dense)

# Compute normalized dot product similarity.
similarity = Dot(axes=1, normalize=True)([query_embed, policy_embed])
# Optionally adjust scaling through an extra dense layer.
output = Dense(1, activation='sigmoid')(similarity)

model = Model(inputs=[customer_input, interaction_input, policy_input], outputs=output)
model.compile(optimizer=Adam(learning_rate=0.001), loss='binary_crossentropy', metrics=['accuracy'])
model.summary()

# ------------------------
# 6. Training Enhancements
# ------------------------
# Early stopping to avoid overfitting.
early_stop = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

# Compute class weights to address imbalance.
class_weights = class_weight.compute_class_weight(class_weight='balanced',
                                                  classes=np.unique(y_train),
                                                  y=y_train)
class_weights_dict = dict(enumerate(class_weights))
print("Class weights:", class_weights_dict)

history = model.fit(
    [X_cust_train, X_interact_train, X_policy_train], y_train,
    validation_split=0.1,
    epochs=20,
    batch_size=32,
    callbacks=[early_stop],
    class_weight=class_weights_dict
)

# ------------------------
# 7. Model Evaluation
# ------------------------
eval_results = model.evaluate([X_cust_test, X_interact_test, X_policy_test], y_test)
print(f"Test Loss: {eval_results[0]:.4f} | Test Accuracy: {eval_results[1]:.4f}")


Purchase distribution:
purchased
0    88
1    12
Name: count, dtype: int64


<ipython-input-26-d120591fa53d>:67: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  policy_features_df[column] = policy_features_df[column].str.replace(',', '').astype(float)
<ipython-input-26-d120591fa53d>:67: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  policy_features_df[column] = policy_features_df[column].str.replace(',', '').astype(float)


Model: "functional_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)              ┃ Output Shape           ┃        Param # ┃ Connected to           ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━┩
│ customer_input            │ (None, 36)             │              0 │ -                      │
│ (InputLayer)              │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ interaction_input         │ (None, 4)              │              0 │ -                      │
│ (InputLayer)              │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ dense_8 (Dense)           │ (None, 64)             │          2,368 │ customer_input[0][0]   │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ dense_9 (Dense)           │ (None, 16)             │             80 │ interaction_input[0][… │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ policy_input (InputLayer) │ (None, 44)             │              0 │ -                      │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ dropout_3 (Dropout)       │ (None, 64)             │              0 │ dense_8[0][0]          │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ dropout_4 (Dropout)       │ (None, 16)             │              0 │ dense_9[0][0]          │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ dense_10 (Dense)          │ (None, 64)             │          2,880 │ policy_input[0][0]     │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ concatenate_2             │ (None, 80)             │              0 │ dropout_3[0][0],       │
│ (Concatenate)             │                        │                │ dropout_4[0][0]        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ dropout_5 (Dropout)       │ (None, 64)             │              0 │ dense_10[0][0]         │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ query_embedding (Dense)   │ (None, 32)             │          2,592 │ concatenate_2[0][0]    │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ policy_embedding (Dense)  │ (None, 32)             │          2,080 │ dropout_5[0][0]        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ dot_1 (Dot)               │ (None, 1)              │              0 │ query_embedding[0][0], │
│                           │                        │                │ policy_embedding[0][0] │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ dense_11 (Dense)          │ (None, 1)              │              2 │ dot_1[0][0]            │
└───────────────────────────┴────────────────────────┴────────────────┴────────────────────────┘

 Total params: 10,002 (39.07 KB)

 Trainable params: 10,002 (39.07 KB)

 Non-trainable params: 0 (0.00 B)

Class weights: {0: 0.5555555555555556, 1: 5.0}
Epoch 1/20
3/3 ━━━━━━━━━━━━━━━━━━━━ 5s 527ms/step - accuracy: 0.8850 - loss: 0.7465 - val_accuracy: 0.8750 - val_loss: 0.6194
Epoch 2/20
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 135ms/step - accuracy: 0.9045 - loss: 0.6697 - val_accuracy: 0.8750 - val_loss: 0.6188
Epoch 3/20
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 118ms/step - accuracy: 0.9045 - loss: 0.6597 - val_accuracy: 0.8750 - val_loss: 0.6199
Epoch 4/20
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 119ms/step - accuracy: 0.9045 - loss: 0.6623 - val_accuracy: 0.8750 - val_loss: 0.6212
Epoch 5/20
3/3 ━━━━━━━━━━━━━━━━━━━━ 1s 124ms/step - accuracy: 0.8928 - loss: 0.6922 - val_accuracy: 0.8750 - val_loss: 0.6226
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 139ms/step - accuracy: 0.8000 - loss: 0.6391
Test Loss: 0.6391 | Test Accuracy: 0.8000


In [ ]:
MODEL_PATH="twotower.h5"
model.save(MODEL_PATH)
print(f"Model saved at {MODEL_PATH}")

Model saved at twotower.h5


# **Two Tower Neural Network for Candidate Generation**

## Importing the libraries

In [1]:
import os
import pandas as pd
import numpy as np
from sklearn.model_selection import GroupShuffleSplit

# ---- Paths (adjust if needed) ----
CUSTOMERS_CSV   = "customers.csv"
INTERACTIONS_CSV= "interactions.csv"
POLICIES_CSV    = "policies_cleaned.csv"  # use cleaned version for numeric money fields

In [2]:
# ---- Load ----
customers   = pd.read_csv(CUSTOMERS_CSV)
interactions= pd.read_csv(INTERACTIONS_CSV)
policies    = pd.read_csv(POLICIES_CSV)

In [3]:
# ---- Sanity: required columns ----
needed_cust = {"customer_id","age","gender","income_bracket","employment_status",
               "marital_status","location_city","policy_ownership_count","credit_score","preferred_policy_type"}
needed_inter= {"customer_id","policy_id","clicked","viewed_duration","comparison_count","abandoned_cart","purchased"}
needed_pol  = {"policy_id","policy_type","risk_category","customer_target_group",
               "sum_assured","premium_amount","policy_duration_years","description","keywords"}

In [4]:
missing = []
for name, need, df in [
    ("customers.csv", needed_cust, customers),
    ("interactions.csv", needed_inter, interactions),
    ("policies_cleaned.csv", needed_pol, policies),
]:
    miss = need - set(df.columns)
    if miss:
        missing.append((name, sorted(miss)))
if missing:
    raise ValueError(f"Missing required columns: {missing}")


In [5]:
# ---- Build a compact nature taxonomy ----
# Keep it small & generic; you can widen later (e.g., bucket customer_target_group)
def make_policy_nature(row):
    t = str(row["policy_type"]).strip().lower()
    r = str(row["risk_category"]).strip().lower()
    return f"{t}__{r}"

policies = policies.copy()
policies["policy_nature"] = policies.apply(make_policy_nature, axis=1)

In [6]:
# Add a lowercase text field combining description + keywords (no embeddings; TF-IDF later)
def coalesce_text(desc, kw):
    d = "" if pd.isna(desc) else str(desc)
    k = "" if pd.isna(kw)   else str(kw)
    return (d + " " + k).strip().lower()

policies["policy_text"] = policies.apply(lambda r: coalesce_text(r["description"], r["keywords"]), axis=1)

In [7]:
# ---- Join interactions with policies & customers (for analysis + building positives) ----
data = (interactions
        .merge(customers, on="customer_id", how="left", validate="many_to_one")
        .merge(policies[["policy_id","policy_nature"]], on="policy_id", how="left", validate="many_to_one"))


In [8]:
# Keep only rows that successfully map to a policy nature
data = data[~data["policy_nature"].isna()].reset_index(drop=True)

# ---- Positives = purchased==1 (we’ll use them to form contrastive pairs) ----
positives = data[data["purchased"] == 1].copy()

if positives.empty:
    raise ValueError("No positive (purchased==1) interactions found; cannot form positive pairs.")

In [9]:
# ---- Nature index mapping (for auxiliary classification head) ----
nature_list = sorted(policies["policy_nature"].unique())
nature_to_idx = {n:i for i,n in enumerate(nature_list)}
idx_to_nature = {i:n for n,i in nature_to_idx.items()}

positives["nature_idx"] = positives["policy_nature"].map(nature_to_idx).astype(int)

print("✅ Loaded data")
print(f"- customers: {customers.shape}, interactions: {interactions.shape}, policies: {policies.shape}")
print(f"- unique policy natures: {len(nature_list)} -> {nature_list[:10]}{'...' if len(nature_list)>10 else ''}")
print(f"- positives (purchased==1): {len(positives)} rows")


✅ Loaded data
- customers: (10, 11), interactions: (100, 8), policies: (49, 12)
- unique policy natures: 12 -> ['annuity__low', 'child plan__low', 'child plan__medium', 'endowment__low', 'health__medium', 'money back__low', 'pension__low', 'retirement__medium', 'savings__low', 'term__low']...
- positives (purchased==1): 12 rows


In [10]:
# ---- Define feature sets (no transforms yet) ----

# Query tower = customer + interaction signals
QUERY_NUMERIC = ["age","policy_ownership_count","credit_score",
                 "clicked","viewed_duration","comparison_count","abandoned_cart"]
QUERY_CATEGORICAL = ["gender","income_bracket","employment_status","marital_status",
                     "location_city","preferred_policy_type"]

# Candidate tower = policy structured + text (text TF-IDF added later)
CAND_NUMERIC = ["sum_assured","premium_amount","policy_duration_years"]
CAND_CATEGORICAL = ["policy_type","risk_category","customer_target_group"]
CAND_TEXT_FIELD = "policy_text"  # from description+keywords

# Keep a slim candidate table to vectorize later
candidates = policies[["policy_id","policy_nature",*CAND_NUMERIC,*CAND_CATEGORICAL,CAND_TEXT_FIELD]].copy()
candidates["nature_idx"] = candidates["policy_nature"].map(nature_to_idx).astype(int)

In [11]:
# ---- Group split (by customer) indices for positives (to avoid leakage) ----
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
groups = positives["customer_id"].values
(train_idx, test_idx), = gss.split(np.zeros(len(positives)), groups=groups)

pos_train = positives.iloc[train_idx].reset_index(drop=True)
pos_test  = positives.iloc[test_idx].reset_index(drop=True)

print(f"- train positives: {len(pos_train)}, test positives: {len(pos_test)}")

- train positives: 9, test positives: 3


In [12]:
# Keep these around for the next snippet
DATA_OBJECTS = {
    "customers": customers,
    "interactions": interactions,
    "policies": policies,
    "data_joined": data,
    "positives": positives,
    "pos_train": pos_train,
    "pos_test": pos_test,
    "candidates": candidates,
    "nature_to_idx": nature_to_idx,
    "idx_to_nature": idx_to_nature,
    "QUERY_NUMERIC": QUERY_NUMERIC,
    "QUERY_CATEGORICAL": QUERY_CATEGORICAL,
    "CAND_NUMERIC": CAND_NUMERIC,
    "CAND_CATEGORICAL": CAND_CATEGORICAL,
    "CAND_TEXT_FIELD": CAND_TEXT_FIELD,
}

## Vectorizers and Preprocessors

In [13]:
import numpy as np
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.feature_extraction.text import TfidfVectorizer

In [14]:
customers        = DATA_OBJECTS["customers"]
interactions     = DATA_OBJECTS["interactions"]
policies         = DATA_OBJECTS["policies"]
data_joined      = DATA_OBJECTS["data_joined"]
positives        = DATA_OBJECTS["positives"]
pos_train        = DATA_OBJECTS["pos_train"]
pos_test         = DATA_OBJECTS["pos_test"]
candidates       = DATA_OBJECTS["candidates"]
nature_to_idx    = DATA_OBJECTS["nature_to_idx"]
idx_to_nature    = DATA_OBJECTS["idx_to_nature"]
QUERY_NUMERIC    = DATA_OBJECTS["QUERY_NUMERIC"]
QUERY_CATEGORICAL= DATA_OBJECTS["QUERY_CATEGORICAL"]
CAND_NUMERIC     = DATA_OBJECTS["CAND_NUMERIC"]
CAND_CATEGORICAL = DATA_OBJECTS["CAND_CATEGORICAL"]
CAND_TEXT_FIELD  = DATA_OBJECTS["CAND_TEXT_FIELD"]

In [15]:
# ---------------------------------
# 1) Query-side preprocessor (fit on train positives only to avoid leakage)
# ---------------------------------
query_num_pipe = Pipeline([
    ("imp", SimpleImputer(strategy="median")),
    ("sc",  StandardScaler())
])
query_cat_pipe = OneHotEncoder(handle_unknown="ignore", sparse_output=False)

query_preproc = ColumnTransformer(
    transformers=[
        ("num", query_num_pipe, QUERY_NUMERIC),
        ("cat", query_cat_pipe, QUERY_CATEGORICAL),
    ],
    remainder="drop",
    sparse_threshold=0.0  # force dense
)

In [16]:
# Build the query-feature DataFrames for train/test
def build_query_df(df_rows: pd.DataFrame) -> pd.DataFrame:
    cols = QUERY_NUMERIC + QUERY_CATEGORICAL
    return df_rows[cols].copy()

Xq_train_df = build_query_df(pos_train)
Xq_test_df  = build_query_df(pos_test)

Xq_train = query_preproc.fit_transform(Xq_train_df).astype("float32")
Xq_test  = query_preproc.transform(Xq_test_df).astype("float32")

print("✅ Query preprocessor fitted")
print("  - Xq_train shape:", Xq_train.shape)
print("  - Xq_test  shape:", Xq_test.shape)

✅ Query preprocessor fitted
  - Xq_train shape: (9, 24)
  - Xq_test  shape: (3, 24)


In [17]:
# ---------------------------------
# 2) Candidate-side preprocessors
#    - Structured: impute + scale + one-hot
#    - Text: TF-IDF on (description + keywords)
# ---------------------------------
cand_num_pipe = Pipeline([
    ("imp", SimpleImputer(strategy="median")),
    ("sc",  StandardScaler())
])
cand_cat_pipe = OneHotEncoder(handle_unknown="ignore", sparse_output=False)

cand_struct_preproc = ColumnTransformer(
    transformers=[
        ("num", cand_num_pipe, CAND_NUMERIC),
        ("cat", cand_cat_pipe, CAND_CATEGORICAL),
    ],
    remainder="drop",
    sparse_threshold=0.0
)

In [18]:
# Fit structured preproc on ALL candidates (this is okay—candidates are "catalog")
Xc_struct = cand_struct_preproc.fit_transform(
    candidates[CAND_NUMERIC + CAND_CATEGORICAL]
).astype("float32")

# TF-IDF for candidate text (no heavy LMs; n-grams help)
tfidf = TfidfVectorizer(
    ngram_range=(1,2),
    max_features=3000,
    min_df=1,
    dtype=np.float32,
    strip_accents="unicode"
)
Xc_text = tfidf.fit_transform(candidates[CAND_TEXT_FIELD])  # sparse
Xc_text = Xc_text.astype(np.float32)


In [19]:
# Combine structured + text
from scipy.sparse import hstack

# Keep as sparse while combining; we can toarray() later if needed
Xc_sparse = hstack([Xc_struct, Xc_text], format="csr")
Xc = Xc_sparse.toarray().astype("float32")  # small catalog -> OK to densify

print("✅ Candidate preprocessors fitted")
print("  - Xc_struct shape:", Xc_struct.shape)
print("  - Xc_text   shape:", Xc_text.shape)
print("  - Xc (combined) shape:", Xc.shape)

✅ Candidate preprocessors fitted
  - Xc_struct shape: (49, 53)
  - Xc_text   shape: (49, 750)
  - Xc (combined) shape: (49, 803)


In [20]:
# ---------------------------------
# 3) Candidate lookup helpers
# ---------------------------------
policy_ids = candidates["policy_id"].tolist()
policy_nature_idx = candidates["nature_idx"].values.astype("int32")
policy_id_to_row = {pid: i for i, pid in enumerate(policy_ids)}

print(f"  - policies in catalog: {len(policy_ids)}")
print(f"  - unique natures: {len(nature_to_idx)}")

  - policies in catalog: 49
  - unique natures: 12


In [21]:
# ---------------------------------
# 4) Save for next snippet
# ---------------------------------
PREP_OBJECTS = {
    "query_preproc": query_preproc,
    "cand_struct_preproc": cand_struct_preproc,
    "tfidf": tfidf,
    "Xq_train": Xq_train,
    "Xq_test": Xq_test,
    "Xc": Xc,
    "policy_ids": policy_ids,
    "policy_nature_idx": policy_nature_idx,
    "policy_id_to_row": policy_id_to_row,
}

print("✅ Preprocessing complete and objects stored in PREP_OBJECTS")

✅ Preprocessing complete and objects stored in PREP_OBJECTS


## Neural Network Training

In [22]:
# ================================================
# Snippet 3: Two-Tower (contrastive + nature aux)
# - Builds train/val batches aligned for in-batch contrastive
# - Keras model with cosine-like logits + nature softmax head
# - Trains with early stopping
# ================================================

import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, Model, regularizers
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping

In [23]:
# Unpack from previous snippets
pos_train        = DATA_OBJECTS["pos_train"]
pos_test         = DATA_OBJECTS["pos_test"]
nature_to_idx    = DATA_OBJECTS["nature_to_idx"]

Xq_train         = PREP_OBJECTS["Xq_train"]
Xq_test          = PREP_OBJECTS["Xq_test"]
Xc               = PREP_OBJECTS["Xc"]
policy_id_to_row = PREP_OBJECTS["policy_id_to_row"]

In [24]:
# ------------------------------------------------
# 1) Align positive candidate rows for each positive interaction
# ------------------------------------------------
def map_policy_rows(df):
    idxs = []
    missing = 0
    for pid in df["policy_id"].tolist():
        j = policy_id_to_row.get(pid, None)
        if j is None:
            missing += 1
            idxs.append(-1)
        else:
            idxs.append(j)
    if missing:
        raise ValueError(f"{missing} policy_id(s) not found in candidate catalog.")
    return np.array(idxs, dtype=np.int32)

train_pos_policy_rows = map_policy_rows(pos_train)
test_pos_policy_rows  = map_policy_rows(pos_test)

Xp_train_pos = Xc[train_pos_policy_rows]  # candidate features aligned with Xq_train
Xp_test_pos  = Xc[test_pos_policy_rows]   # candidate features aligned with Xq_test

y_nature_train = pos_train["nature_idx"].values.astype("int32")
y_nature_test  = pos_test["nature_idx"].values.astype("int32")

# Basic sanity checks
assert Xq_train.shape[0] == Xp_train_pos.shape[0] == y_nature_train.shape[0]
assert Xq_test.shape[0]  == Xp_test_pos.shape[0]  == y_nature_test.shape[0]

n_train = Xq_train.shape[0]
n_test  = Xq_test.shape[0]

if n_train < 2:
    raise ValueError("Need at least 2 training positives for in-batch contrastive learning.")

In [25]:
# ------------------------------------------------
# 2) Batch generator: returns aligned (q, p) and labels
#    - contrastive labels are [0..B-1] so each query's positive is at same index
#    - nature labels are per-query nature indices
# ------------------------------------------------
class ContrastiveNatureSequence(tf.keras.utils.Sequence):
    def __init__(self, Xq, Xp, y_nat, batch_size=16, shuffle=True):
        self.Xq = Xq
        self.Xp = Xp
        self.y  = y_nat
        self.bs = max(2, min(batch_size, len(self.Xq)))  # ensure >=2, cap to dataset
        self.shuffle = shuffle
        self.idx = np.arange(len(self.Xq))
        self.on_epoch_end()

    def __len__(self):
        return int(np.ceil(len(self.idx) / self.bs))

    def on_epoch_end(self):
        if self.shuffle:
            np.random.shuffle(self.idx)

    def __getitem__(self, i):
        sl = slice(i*self.bs, (i+1)*self.bs)
        ids = self.idx[sl]
        q = self.Xq[ids]
        p = self.Xp[ids]
        y_nat = self.y[ids]
        # Labels for contrastive: 0..B-1
        y_con = np.arange(q.shape[0], dtype=np.int32)
        return {"q_in": q, "p_in": p}, {"contrastive": y_con, "nature": y_nat}

# Create train/val sequences
BATCH_SIZE = 16
train_seq = ContrastiveNatureSequence(Xq_train, Xp_train_pos, y_nature_train, batch_size=BATCH_SIZE, shuffle=True)
val_seq   = ContrastiveNatureSequence(Xq_test,  Xp_test_pos,  y_nature_test,  batch_size=BATCH_SIZE, shuffle=False)


In [26]:
# ------------------------------------------------
# 3) Build two-tower model
#    - query tower: customer+interaction features
#    - candidate tower: policy features (structured + TF-IDF)
#    - outputs:
#       * "contrastive": BxB logits (InfoNCE via SparseCategoricalCrossentropy)
#       * "nature":      per-query nature logits (auxiliary head)
# ------------------------------------------------
tf.random.set_seed(42)

q_dim = Xq_train.shape[1]
p_dim = Xp_train_pos.shape[1]
num_natures = len(nature_to_idx)
EMBED_DIM = 32
L2 = 1e-5
TEMPERATURE = tf.constant(0.07, dtype=tf.float32)  # lower = sharper softmax

# Inputs
q_in = layers.Input(shape=(q_dim,), name="q_in")
p_in = layers.Input(shape=(p_dim,), name="p_in")


In [28]:
# Query tower
xq = layers.Dense(64, activation="relu", kernel_regularizer=regularizers.l2(L2))(q_in)
xq = layers.Dropout(0.2)(xq)
q_latent = layers.Dense(EMBED_DIM, activation="relu", kernel_regularizer=regularizers.l2(L2))(xq)
q_emb = layers.Lambda(lambda t: tf.nn.l2_normalize(t, axis=-1), name="q_emb")(q_latent)

# Nature auxiliary head (from pre-norm latent)
nature_logits = layers.Dense(num_natures, name="nature")(q_latent)

# Candidate tower
xp = layers.Dense(128, activation="relu", kernel_regularizer=regularizers.l2(L2))(p_in)
xp = layers.Dropout(0.2)(xp)
p_latent = layers.Dense(EMBED_DIM, activation="relu", kernel_regularizer=regularizers.l2(L2))(xp)
p_emb = layers.Lambda(lambda t: tf.nn.l2_normalize(t, axis=-1), name="p_emb")(p_latent)

# In-batch similarity logits: (B,D) · (B,D)^T -> (B,B); scale by 1/T
def sim_logits(tensors):
    q, p = tensors
    return tf.matmul(q, p, transpose_b=True) / TEMPERATURE

contrastive_logits = layers.Lambda(sim_logits, name="contrastive")([q_emb, p_emb])

model = Model(inputs=[q_in, p_in], outputs=[contrastive_logits, nature_logits], name="TwoTowerNature")

In [29]:
# Losses & metrics
losses = {
    "contrastive": tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    "nature":      tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
}
metrics = {
    "contrastive": [tf.keras.metrics.SparseCategoricalAccuracy(name="acc")],
    "nature":      [tf.keras.metrics.SparseCategoricalAccuracy(name="acc")],
}
loss_weights = {"contrastive": 1.0, "nature": 0.3}

model.compile(optimizer=Adam(1e-3), loss=losses, loss_weights=loss_weights, metrics=metrics)
model.summary()

Model: "TwoTowerNature"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ q_in (InputLayer)   │ (None, 24)        │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ p_in (InputLayer)   │ (None, 803)       │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_4 (Dense)     │ (None, 64)        │      1,600 │ q_in[0][0]        │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_6 (Dense)     │ (None, 128)       │    102,912 │ p_in[0][0]        │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_2 (Dropout) │ (None, 64)        │          0 │ dense_4[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_3 (Dropout) │ (None, 128)       │          0 │ dense_6[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_5 (Dense)     │ (None, 32)        │      2,080 │ dropout_2[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_7 (Dense)     │ (None, 32)        │      4,128 │ dropout_3[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ q_emb (Lambda)      │ (None, 32)        │          0 │ dense_5[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ p_emb (Lambda)      │ (None, 32)        │          0 │ dense_7[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ contrastive         │ (None, None)      │          0 │ q_emb[0][0],      │
│ (Lambda)            │                   │            │ p_emb[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ nature (Dense)      │ (None, 12)        │        396 │ dense_5[0][0]     │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 111,116 (434.05 KB)

 Trainable params: 111,116 (434.05 KB)

 Non-trainable params: 0 (0.00 B)

In [30]:
# ------------------------------------------------
# 4) Train
# ------------------------------------------------
callbacks = [
    EarlyStopping(monitor="val_loss", patience=10, restore_best_weights=True, verbose=1)
]

EPOCHS = 200  # early stopping will kick in
history = model.fit(
    train_seq,
    validation_data=val_seq,
    epochs=EPOCHS,
    callbacks=callbacks,
    verbose=1
)

Epoch 1/200


/usr/local/lib/python3.11/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


1/1 ━━━━━━━━━━━━━━━━━━━━ 7s 7s/step - contrastive_acc: 0.0000e+00 - contrastive_loss: 3.3360 - loss: 4.0566 - nature_acc: 0.0000e+00 - nature_loss: 2.3901 - val_contrastive_acc: 0.3333 - val_contrastive_loss: 1.4027 - val_loss: 2.1162 - val_nature_acc: 0.0000e+00 - val_nature_loss: 2.3667
Epoch 2/200
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 71ms/step - contrastive_acc: 0.1111 - contrastive_loss: 2.5043 - loss: 3.1958 - nature_acc: 0.2222 - nature_loss: 2.2934 - val_contrastive_acc: 0.3333 - val_contrastive_loss: 1.4274 - val_loss: 2.1411 - val_nature_acc: 0.0000e+00 - val_nature_loss: 2.3675
Epoch 3/200
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 62ms/step - contrastive_acc: 0.0000e+00 - contrastive_loss: 2.3575 - loss: 3.0356 - nature_acc: 0.0000e+00 - nature_loss: 2.2488 - val_contrastive_acc: 0.3333 - val_contrastive_loss: 1.4470 - val_loss: 2.1613 - val_nature_acc: 0.0000e+00 - val_nature_loss: 2.3696
Epoch 4/200
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 62ms/step - contrastive_acc: 0.2222 - contrastive_loss: 1.9213 - loss:

In [31]:
# ------------------------------------------------
# 5) Quick validation metrics
# ------------------------------------------------
val_metrics = model.evaluate(val_seq, verbose=0)
print("Validation metrics:")
for n, v in zip(model.metrics_names, val_metrics):
    print(f"  {n}: {v:.4f}")

Validation metrics:
  loss: 2.1162
  compile_metrics: 1.4027
  contrastive_loss: 2.3667
  nature_loss: 0.3333


In [32]:
# Keep encoders for the next step (export & retrieval demo)
query_encoder   = Model(inputs=q_in, outputs=q_emb, name="query_encoder")
policy_encoder  = Model(inputs=p_in, outputs=p_emb, name="policy_encoder")

print("✅ Training complete. Query & policy encoders are ready.")

✅ Training complete. Query & policy encoders are ready.


## Execution and model saving

In [39]:
# ================================================
# Snippet 4: Export + Top-K Retrieval Demo (Keras 3 safe)
# - Saves preprocessors (joblib) and encoders (.keras)
# - Builds nature-probability submodel safely
# - Precomputes policy embeddings
# - Cosine Top-K recommender + brief demo
# ================================================

import os, json, numpy as np, pandas as pd
from pathlib import Path
import joblib
import tensorflow as tf
from tensorflow.keras import Model, layers

# --------- Unpack from previous snippets ----------
# From Snippet 1
policies_full      = DATA_OBJECTS["policies"]            # full policies with 'policy_nature', text, etc.
pos_test           = DATA_OBJECTS["pos_test"]
pos_train          = DATA_OBJECTS["pos_train"]
idx_to_nature      = DATA_OBJECTS["idx_to_nature"]
QUERY_NUMERIC      = DATA_OBJECTS["QUERY_NUMERIC"]
QUERY_CATEGORICAL  = DATA_OBJECTS["QUERY_CATEGORICAL"]

# From Snippet 2
query_preproc       = PREP_OBJECTS["query_preproc"]
cand_struct_preproc = PREP_OBJECTS["cand_struct_preproc"]
tfidf               = PREP_OBJECTS["tfidf"]
Xc                  = PREP_OBJECTS["Xc"]
policy_ids          = PREP_OBJECTS["policy_ids"]

In [40]:
# --------- Create artifacts folder ----------
ART_DIR = Path("artifacts")
ART_DIR.mkdir(parents=True, exist_ok=True)

# --------- Save preprocessors ----------
joblib.dump(query_preproc,       ART_DIR / "query_preproc.joblib")
joblib.dump(cand_struct_preproc, ART_DIR / "cand_struct_preproc.joblib")
joblib.dump(tfidf,               ART_DIR / "tfidf.joblib")
print("✅ Saved preprocessors (joblib)")

# --------- Save encoders (Keras 3 format) ----------
query_encoder_path  = ART_DIR / "query_encoder.keras"
policy_encoder_path = ART_DIR / "policy_encoder.keras"
query_encoder.save(query_encoder_path, include_optimizer=False)
policy_encoder.save(policy_encoder_path, include_optimizer=False)
print("✅ Saved encoders (.keras):")
print("   -", query_encoder_path)
print("   -", policy_encoder_path)

✅ Saved preprocessors (joblib)
✅ Saved encoders (.keras):
   - artifacts/query_encoder.keras
   - artifacts/policy_encoder.keras


In [41]:
# --------- Build nature-probability submodel safely (Keras 3) ----------
# 'nature' Dense head exists in the main model (from Snippet 3).
try:
    nature_logits = model.get_layer("nature").output
except ValueError as e:
    raise ValueError("Could not find a layer named 'nature' in the model. "
                     "Ensure Snippet 3 built the model with a 'nature' head.") from e

nature_probs  = layers.Activation("softmax", name="nature_probs")(nature_logits)

# Use the model's first input (this is 'q_in' per Snippet 3)
q_input_tensor = model.inputs[0]
query_nature_model = Model(inputs=q_input_tensor, outputs=nature_probs, name="query_nature_model")

model_nature_path = ART_DIR / "query_nature_model.keras"
query_nature_model.save(model_nature_path, include_optimizer=False)
print("✅ Saved query nature-probability model (.keras):", model_nature_path)

# Save idx_to_nature mapping for readable explanations
with open(ART_DIR / "idx_to_nature.json", "w") as f:
    json.dump({int(k): v for k, v in idx_to_nature.items()}, f, indent=2)
print("✅ Saved idx_to_nature map")

✅ Saved query nature-probability model (.keras): artifacts/query_nature_model.keras
✅ Saved idx_to_nature map


In [42]:
# --------- Precompute policy embeddings (catalog) ----------
policy_embeds = policy_encoder.predict(Xc, batch_size=64, verbose=0)  # L2-normalized already
np.save(ART_DIR / "policy_embeds.npy", policy_embeds)
print("✅ Precomputed policy embeddings:", policy_embeds.shape)

# --------- Build catalog aligned with embeddings ----------
pol_lookup = policies_full.set_index("policy_id")
catalog_rows = []
for pid in policy_ids:
    row = pol_lookup.loc[pid]
    catalog_rows.append({
        "policy_id": pid,
        "policy_name": row.get("policy_name", ""),
        "policy_nature": row.get("policy_nature", ""),
        "description": row.get("description", ""),
        "keywords": row.get("keywords", ""),
        "sum_assured": row.get("sum_assured", np.nan),
        "premium_amount": row.get("premium_amount", np.nan),
        "policy_duration_years": row.get("policy_duration_years", np.nan),
        "policy_type": row.get("policy_type", ""),
        "risk_category": row.get("risk_category", ""),
        "customer_target_group": row.get("customer_target_group", "")
    })
policy_catalog = pd.DataFrame(catalog_rows)
catalog_path = ART_DIR / "policy_catalog.csv"
policy_catalog.to_csv(catalog_path, index=False)
print("✅ Saved policy catalog CSV:", policy_catalog.shape, "->", catalog_path)

✅ Precomputed policy embeddings: (49, 32)
✅ Saved policy catalog CSV: (49, 11) -> artifacts/policy_catalog.csv


In [43]:
# --------- Helper functions ----------
def build_query_embedding(raw_df: pd.DataFrame) -> np.ndarray:
    """
    raw_df: DataFrame with exactly one row containing QUERY_NUMERIC + QUERY_CATEGORICAL
    returns: (1, D) L2-normalized embedding
    """
    xq = query_preproc.transform(raw_df[QUERY_NUMERIC + QUERY_CATEGORICAL]).astype("float32")
    q_emb = query_encoder.predict(xq, verbose=0)
    return q_emb

def predict_query_nature(raw_df: pd.DataFrame, topn=3):
    xq = query_preproc.transform(raw_df[QUERY_NUMERIC + QUERY_CATEGORICAL]).astype("float32")
    probs = query_nature_model.predict(xq, verbose=0)[0]
    order = np.argsort(-probs)[:topn]
    return [(idx_to_nature[i], float(probs[i])) for i in order]

def topk_policies_for_query(q_emb: np.ndarray, K=5) -> pd.DataFrame:
    """
    q_emb: (1, D) query embedding (L2-normalized)
    returns: DataFrame of top-K policies with cosine scores
    """
    sims = (q_emb @ policy_embeds.T).ravel()  # cosine because embeddings are L2-normalized
    K = int(K)
    K = min(max(K, 1), len(sims))
    top_idx = np.argpartition(-sims, K-1)[:K]
    top_idx = top_idx[np.argsort(-sims[top_idx])]
    out = policy_catalog.iloc[top_idx].copy()
    out.insert(1, "score", sims[top_idx])
    return out.reset_index(drop=True)

In [44]:
# --------- Demo on one held-out example ----------
demo_src = pos_test if not pos_test.empty else pos_train
demo_row = demo_src.iloc[[0]]
demo_q   = demo_row[QUERY_NUMERIC + QUERY_CATEGORICAL].copy()

In [45]:
# Nature explanation
top_natures = predict_query_nature(demo_q, topn=3)
print("\n🔎 Predicted top-3 natures for this query:")
for n, p in top_natures:
    print(f"  - {n:30s}  p={p:.3f}")


🔎 Predicted top-3 natures for this query:
  - ulip__medium                    p=0.121
  - term__low                       p=0.111
  - money back__low                 p=0.098


In [46]:
# Top-K retrieval
q_emb = build_query_embedding(demo_q)
topK = topk_policies_for_query(q_emb, K=5)
topK["nature_match"] = topK["policy_nature"].isin([n for n, _ in top_natures])

In [47]:
display_cols = [
    "policy_id","score","policy_name","policy_nature","nature_match",
    "policy_type","risk_category","customer_target_group",
    "sum_assured","premium_amount","policy_duration_years"
]
print("\n⭐ Top-5 policy recommendations:")
print(topK[display_cols].to_string(index=False))


⭐ Top-5 policy recommendations:
 policy_id    score                      policy_name   policy_nature  nature_match policy_type risk_category                    customer_target_group  sum_assured  premium_amount  policy_duration_years
        30 0.612389        SBI Life - Smart Humsafar  endowment__low         False   Endowment           Low Married couples seeking joint life cover      1000000           15000                     25
        10 0.608808     SBI Life - Smart Bachat Plus  endowment__low         False   Endowment           Low             Budget-Conscious Individuals       500000            5100                     20
        16 0.595773 SBI Life - Smart Champ Insurance child plan__low         False  Child Plan           Low                                  Parents       500000            6000                     20
        49 0.582126        SBI Life - Smart Humsafar  endowment__low         False   Endowment           Low Married couples seeking joint life cover      1000

In [48]:
# Save demo output
demo_csv = ART_DIR / "demo_topK.csv"
topK.to_csv(demo_csv, index=False)

print("\n✅ Artifacts written to:", ART_DIR.resolve())
for p in sorted(ART_DIR.iterdir()):
    print(" -", p.name)


✅ Artifacts written to: /content/artifacts
 - cand_struct_preproc.joblib
 - demo_topK.csv
 - idx_to_nature.json
 - policy_catalog.csv
 - policy_embeds.npy
 - policy_encoder.keras
 - query_encoder.keras
 - query_nature_model.keras
 - query_preproc.joblib
 - tfidf.joblib


## Saving h5 model

In [49]:
# ================================
# Save full model + encoders (Keras 3) and H5 variants
# ================================
from pathlib import Path
import joblib, numpy as np
import tensorflow as tf

ART_DIR = Path("artifacts")
ART_DIR.mkdir(exist_ok=True, parents=True)

# 1) Save preprocessors (already done in Snippet 4, safe to re-run)
joblib.dump(PREP_OBJECTS["query_preproc"],        ART_DIR / "query_preproc.joblib")
joblib.dump(PREP_OBJECTS["cand_struct_preproc"],  ART_DIR / "cand_struct_preproc.joblib")
joblib.dump(PREP_OBJECTS["tfidf"],                ART_DIR / "tfidf.joblib")

# 2) Save encoders (already done, safe to re-run)
query_encoder.save(ART_DIR / "query_encoder.keras", include_optimizer=False)
policy_encoder.save(ART_DIR / "policy_encoder.keras", include_optimizer=False)

# 3) Save the *full* multitask model (recommended format)
FULL_KERAS = ART_DIR / "two_tower_full.keras"
model.save(FULL_KERAS, include_optimizer=False)
print("✅ Saved full model (native Keras):", FULL_KERAS)

# 4) Try saving the full model as H5 (may fail with Lambda layers)
FULL_H5 = ART_DIR / "two_tower_full.h5"
try:
    model.save(FULL_H5, include_optimizer=False)  # legacy HDF5 format
    print("✅ Saved full model (H5):", FULL_H5)
except Exception as e:
    print("⚠️ Could not save full model as .h5 due to Lambda/custom ops:", e)
    # Fallback: at least save weights to H5
    WEIGHTS_H5 = ART_DIR / "two_tower_weights.h5"
    model.save_weights(WEIGHTS_H5)
    print("✅ Saved weights only (H5):", WEIGHTS_H5)

# 5) (Optional) also save the nature-probabilities submodel again for convenience
from tensorflow.keras import Model, layers
nature_probs = layers.Activation("softmax", name="nature_probs")(model.get_layer("nature").output)
query_nature_model = Model(inputs=model.inputs[0], outputs=nature_probs, name="query_nature_model")
query_nature_model.save(ART_DIR / "query_nature_model.keras", include_optimizer=False)
print("✅ Saved query nature-probability model (.keras)")


✅ Saved full model (native Keras): artifacts/two_tower_full.keras
✅ Saved full model (H5): artifacts/two_tower_full.h5
✅ Saved query nature-probability model (.keras)
